# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tabassumrafiq/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [6]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv


In [7]:
print("Dataset shape:", df.shape)

print("\nRequired columns:")
required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "avg_position"
]

for col in required_columns:
    print(col, ":", col in df.columns)

Dataset shape: (30000, 44)

Required columns:
content_id : True
client_id : True
impressions_90d : True
avg_position : True


In [9]:
assert "impressions_90d" in df.columns
assert "avg_position" in df.columns
assert "content_id" in df.columns
assert "client_id" in df.columns

print("All required columns are available.")

All required columns are available.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My Rule and Its Reason Codes

### Rule

I will prioritize content for review when it has meaningful observed search visibility together with a weaker average search position.

The baseline combines observed 90-day impressions with average search position. Higher impressions represent greater observed search visibility, while a higher average-position value represents a weaker observed ranking position.

This is a directional decision-support rule, not proof that the content must be refreshed.

### Reason Codes

- `VISIBLE_POSITION_REVIEW` — the content has a high combined baseline score from observed impressions and weaker average position.
- `NO_HIGH_PRIORITY_SIGNAL` — the content does not meet the review threshold under this baseline.

. Prepare the Baseline Data

I use only observed historical signals available in the dataset. No future performance, trend label, or refresh outcome is used to calculate the score.

In [10]:
baseline_df = df[
    [
        "client_id",
        "content_id",
        "impressions_90d",
        "avg_position"
    ]
].copy()

print("Initial baseline rows:", len(baseline_df))

display(baseline_df.head())

Initial baseline rows: 30000


,client_id,content_id,impressions_90d,avg_position
0,client_f369cb89fc,content_304f48230142,3803,10.6
1,client_4e07408562,content_a1fb4e703a9e,15320,20.3
2,client_7f2253d7e2,content_9aa793d4d895,12581,36.5
3,client_19581e27de,content_331d6c4de07b,11751,6.2
4,client_3fdba35f04,content_d99b7a2d90ca,19140,44.0


In [11]:
baseline_df = baseline_df.dropna(
    subset=[
        "impressions_90d",
        "avg_position"
    ]
).copy()

baseline_df = baseline_df[
    (baseline_df["impressions_90d"] >= 0) &
    (baseline_df["avg_position"] > 0)
].copy()

print("Valid baseline rows:", len(baseline_df))

print("\nMissing values:")
print(
    baseline_df[
        ["impressions_90d", "avg_position"]
    ].isna().sum()
)

Valid baseline rows: 28795

Missing values:
impressions_90d    0
avg_position       0
dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

 Build the Ranked Queue

The baseline converts impressions and average position into percentile ranks.

Higher impressions increase the score.

Higher average position also increases the score because a higher position value represents weaker observed ranking performance.

The two percentile ranks receive equal weight.

In [12]:
baseline_df["impression_rank"] = (
    baseline_df["impressions_90d"]
    .rank(pct=True)
)

baseline_df["position_rank"] = (
    baseline_df["avg_position"]
    .rank(pct=True)
)

baseline_df["score"] = (
    baseline_df["impression_rank"] +
    baseline_df["position_rank"]
)

baseline_df["score"] = baseline_df["score"].round(6)

print("Score created successfully.")

display(
    baseline_df[
        [
            "content_id",
            "impressions_90d",
            "avg_position",
            "impression_rank",
            "position_rank",
            "score"
        ]
    ].head()
)

Score created successfully.


,content_id,impressions_90d,avg_position,impression_rank,position_rank,score
0,content_304f48230142,3803,10.6,0.746970,0.472669,1.219639
1,content_a1fb4e703a9e,15320,20.3,0.917486,0.707536,1.625022
2,content_9aa793d4d895,12581,36.5,0.899253,0.894357,1.793610
3,content_331d6c4de07b,11751,6.2,0.892065,0.216739,1.108804
4,content_d99b7a2d90ca,19140,44.0,0.934103,0.932853,1.866956



. Assign Review Actions

The top 10% of the baseline score distribution is marked as `REVIEW`.

All remaining items are marked as `KEEP`.

This threshold is a prioritization rule, not a statement that the remaining content is healthy.

In [13]:
review_threshold = baseline_df["score"].quantile(0.90)

baseline_df["action"] = np.where(
    baseline_df["score"] >= review_threshold,
    "REVIEW",
    "KEEP"
)

baseline_df["reason_code"] = np.where(
    baseline_df["action"] == "REVIEW",
    "VISIBLE_POSITION_REVIEW",
    "NO_HIGH_PRIORITY_SIGNAL"
)

baseline_df["confidence_note"] = np.where(
    baseline_df["action"] == "REVIEW",
    "Directional: observed visibility and weaker position both contribute to the score.",
    "Lower priority under this baseline; not proof that no refresh is needed."
)

print(
    "Review threshold:",
    round(review_threshold, 6)
)

print("\nAction counts:")
display(
    baseline_df["action"].value_counts()
)

Review threshold: 1.487057

Action counts:


,count
action,
KEEP,25915
REVIEW,2880


. Ranked Queue

The complete dataset is ranked from highest to lowest baseline score.

In [14]:
baseline_df = baseline_df.sort_values(
    by=[
        "score",
        "impressions_90d"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)

baseline_df["rank"] = np.arange(
    1,
    len(baseline_df) + 1
)

print("Total ranked items:", len(baseline_df))

display(
    baseline_df[
        [
            "rank",
            "content_id",
            "impressions_90d",
            "avg_position",
            "score",
            "action",
            "reason_code"
        ]
    ].head(20)
)

Total ranked items: 28795


,rank,content_id,impressions_90d,avg_position,score,action,reason_code
0,1,content_a023517539fe,214047,85.8,1.997291,REVIEW,VISIBLE_POSITION_REVIEW
1,2,content_bcf8e8e2280d,32820,83.1,1.964855,REVIEW,VISIBLE_POSITION_REVIEW
2,3,content_109f8f7c9d39,90476,54.4,1.958760,REVIEW,VISIBLE_POSITION_REVIEW
3,4,content_62abc4bd66be,31364,68.5,1.952509,REVIEW,VISIBLE_POSITION_REVIEW
4,5,content_df71843dcd17,27334,76.4,1.951867,REVIEW,VISIBLE_POSITION_REVIEW
5,6,content_54baba704595,130617,47.0,1.941309,REVIEW,VISIBLE_POSITION_REVIEW
6,7,content_fb66dd8f4629,32518,56.5,1.936864,REVIEW,VISIBLE_POSITION_REVIEW
7,8,content_fb4bf6555c79,84093,45.6,1.931707,REVIEW,VISIBLE_POSITION_REVIEW
8,9,content_150f89b1d73b,83490,45.0,1.929050,REVIEW,VISIBLE_POSITION_REVIEW
9,10,content_5096a9d25fe5,49158,46.6,1.924744,REVIEW,VISIBLE_POSITION_REVIEW


. Export the Ranked Queue

The ranked baseline queue is saved to `work/outputs/baseline_action_score.csv`.

In [15]:
output_path = Path(
    "work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

output_columns = [
    "rank",
    "client_id",
    "content_id",
    "impressions_90d",
    "avg_position",
    "score",
    "action",
    "reason_code",
    "confidence_note"
]

baseline_df[
    output_columns
].to_csv(
    output_path,
    index=False
)

print("CSV written successfully:")
print(output_path)

print("Rows written:", len(baseline_df))

CSV written successfully:
work/outputs/baseline_action_score.csv
Rows written: 28795


In [16]:
saved_queue = pd.read_csv(output_path)

print("CSV exists:", output_path.exists())
print("CSV rows:", len(saved_queue))

print("\nCSV columns:")
print(saved_queue.columns.tolist())

display(saved_queue.head(20))

CSV exists: True
CSV rows: 28795

CSV columns:
['rank', 'client_id', 'content_id', 'impressions_90d', 'avg_position', 'score', 'action', 'reason_code', 'confidence_note']


,rank,client_id,content_id,impressions_90d,avg_position,score,action,reason_code,confidence_note
0,1,client_6208ef0f77,content_a023517539fe,214047,85.8,1.997291,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...
1,2,client_6208ef0f77,content_bcf8e8e2280d,32820,83.1,1.964855,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...
2,3,client_6208ef0f77,content_109f8f7c9d39,90476,54.4,1.958760,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...
3,4,client_4e07408562,content_62abc4bd66be,31364,68.5,1.952509,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...
4,5,client_8527a891e2,content_df71843dcd17,27334,76.4,1.951867,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...
5,6,client_6208ef0f77,content_54baba704595,130617,47.0,1.941309,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...
6,7,client_6208ef0f77,content_fb66dd8f4629,32518,56.5,1.936864,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...
7,8,client_6208ef0f77,content_fb4bf6555c79,84093,45.6,1.931707,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...
8,9,client_7f2253d7e2,content_150f89b1d73b,83490,45.0,1.929050,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...
9,10,client_6208ef0f77,content_5096a9d25fe5,49158,46.6,1.924744,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

. Top-20 Review

The top 20 items are treated as decision-support candidates.

For each item, the action and reason code explain how the baseline prioritized it.

The recommendation can be wrong if the observed metrics do not represent current content quality, search intent, competition, or other important context.

In [17]:
top20 = baseline_df.head(20).copy()

top20["what_would_make_it_wrong"] = (
    "Observed impressions or average position may not reflect "
    "current content quality, search intent, competition, "
    "or another factor not represented by this baseline."
)

top20_review = top20[
    [
        "rank",
        "content_id",
        "impressions_90d",
        "avg_position",
        "score",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

,rank,content_id,impressions_90d,avg_position,score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_a023517539fe,214047,85.8,1.997291,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...,Observed impressions or average position may n...
1,2,content_bcf8e8e2280d,32820,83.1,1.964855,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...,Observed impressions or average position may n...
2,3,content_109f8f7c9d39,90476,54.4,1.958760,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...,Observed impressions or average position may n...
3,4,content_62abc4bd66be,31364,68.5,1.952509,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...,Observed impressions or average position may n...
4,5,content_df71843dcd17,27334,76.4,1.951867,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...,Observed impressions or average position may n...
5,6,content_54baba704595,130617,47.0,1.941309,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...,Observed impressions or average position may n...
6,7,content_fb66dd8f4629,32518,56.5,1.936864,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...,Observed impressions or average position may n...
7,8,content_fb4bf6555c79,84093,45.6,1.931707,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...,Observed impressions or average position may n...
8,9,content_150f89b1d73b,83490,45.0,1.929050,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...,Observed impressions or average position may n...
9,10,content_5096a9d25fe5,49158,46.6,1.924744,REVIEW,VISIBLE_POSITION_REVIEW,Directional: observed visibility and weaker po...,Observed impressions or average position may n...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 8. Weak Picks

A high baseline score does not prove that an item needs refreshing.

The highest-ranked items can be weak picks when their score is driven by only the two observed signals used by this baseline.

In [19]:
baseline_df["signal_gap"] = (
    baseline_df["impression_rank"] -
    baseline_df["position_rank"]
).abs()

weak_picks = (
    baseline_df[
        baseline_df["rank"] <= 20
    ]
    .sort_values(
        "signal_gap",
        ascending=False
    )
    .head(5)
)

print("Possible weak picks from the top-20:")

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "impressions_90d",
            "avg_position",
            "score",
            "action",
            "reason_code",
            "signal_gap"
        ]
    ]
)

Possible weak picks from the top-20:


,rank,content_id,impressions_90d,avg_position,score,action,reason_code,signal_gap
17,18,content_370de6e8e035,114389,39.7,1.908022,REVIEW,VISIBLE_POSITION_REVIEW,0.083018
14,15,content_88d367c507a3,130932,40.1,1.911443,REVIEW,VISIBLE_POSITION_REVIEW,0.082375
13,14,content_b51e2e4d22ff,91795,41.2,1.913422,REVIEW,VISIBLE_POSITION_REVIEW,0.072964
16,17,content_11267b0c9023,57314,42.0,1.909047,REVIEW,VISIBLE_POSITION_REVIEW,0.061087
8,9,content_150f89b1d73b,83490,45.0,1.929050,REVIEW,VISIBLE_POSITION_REVIEW,0.054697


### Weak Pick Review

The highest-ranked items should be treated as review candidates rather than confirmed refresh needs.

A high score can result from strong observed visibility combined with a weaker observed average position. The baseline does not measure content quality, search intent, competition, or whether a refresh would improve performance.

## 9. Leakage Check

The baseline score uses only observed `impressions_90d` and `avg_position`.

No future-window performance, trend label, refresh outcome, or product flag is used to calculate the score.

In [20]:
score_features = [
    "impressions_90d",
    "avg_position"
]

leakage_terms = [
    "trend",
    "last30",
    "future",
    "outcome",
    "label",
    "needs_refresh",
    "product_flag"
]

possible_leaks = [
    col
    for col in score_features
    if any(
        term in col.lower()
        for term in leakage_terms
    )
]

print("Score features:")
print(score_features)

print("\nPossible leakage columns:")
print(possible_leaks)

assert len(possible_leaks) == 0

print("\nLeakage check passed.")
print(
    "No future-window, label-derived, "
    "or product-flag fields are used in the score."
)

Score features:
['impressions_90d', 'avg_position']

Possible leakage columns:
[]

Leakage check passed.
No future-window, label-derived, or product-flag fields are used in the score.


## 10. Final Conclusion

This baseline provides a transparent ranking of content review candidates using two observed search signals: 90-day impressions and average search position.

The highest-ranked items should be reviewed by a human before any refresh decision is made.

The score is directional decision-support and should not be interpreted as proof that content is declining or that refreshing it will improve performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### ML-07 completion check

- [x] Plain-language baseline rule is documented.
- [x] Reason codes are documented.
- [x] Score uses transparent hand-written logic.
- [x] Ranked queue is created.
- [x] `work/outputs/baseline_action_score.csv` is written by the notebook.
- [x] Top-20 review is generated from the ranked queue.
- [x] Weak-pick candidates are inspected.
- [x] Leakage check is included.
- [ ] Notebook has been run top to bottom without errors.
- [ ] Final top-20 review has been manually checked.
- [ ] Notebook is committed to `work/notebooks/w04_baseline_score.ipynb`.